# Lesson 30 Lab — End-to-End Project: A Serviceable INT4 Plan for a 70B-Class Model

**Puzzle:** What evidence is required to move from a four-bit checkpoint to a serviceable 70B deployment plan?

The saved outputs were generated by executing every code cell on the recorded RTX 5090. Run all cells to regenerate the evidence on your own CUDA GPU.

## 0. Predict before running

Write down: (1) the expected direction, (2) the mechanism, (3) the observation that would reverse your prediction, and (4) the evidence level required for the claim.

## 1. Theory — objects and data flow

A serviceable 70B plan joins model revision, quantization/calibration, hardware topology, engine, cache policy, quality suite, workload/SLO, capacity/cost, observability, ownership, and rollback.

### Core mechanism

The project is a gate graph rather than one conversion command: memory feasibility enables engine build; engine evidence enables quality/performance tests; only passing all critical gates enables canary.

In [1]:
from pathlib import Path
import json
import sys
import torch

chapter_rel = Path("chapters/01-mixed-precision-int4")
repo_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / chapter_rel / "support" / "lab_common.py").exists()
)
sys.path.insert(0, str(repo_root / chapter_rel / "support"))
from lab_common import (base_result, cuda_benchmark, environment_record,
                        error_metrics, require_cuda, save_result,
                        symmetric_quantize)

lesson_dir = repo_root / chapter_rel / "30-end-to-end-70b-plan"
device = require_cuda()
torch.manual_seed(2026 + 30)
environment = environment_record()
print(json.dumps(environment, indent=2))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "gpu_memory_gib": 31.358,
  "python": "3.12.13",
  "torch": "2.12.0",
  "cuda_runtime": "13.0"
}


## 2. Connect theory to the experiment

### Engineering trade-off

Ideal INT4 arithmetic can suggest single-GPU fit while metadata, unquantized layers, workspaces, and KV cache invalidate it. A multi-GPU plan may fit but violate latency or cost.

### What this code tests

The final lab combines live capacity arithmetic and a small mixed-bit CUDA probe, then returns `not_ready_for_service` because the 70B engine, quality, and service gates were not executed.

**Experiment:** Combine live GPU capacity, a small CUDA mixed-bit quality probe, and a gate matrix to produce a bounded 70B deployment decision.

**Declared evidence label:** `capacity-model`. Check that the shapes, controlled variables, and units match the theoretical question before executing.

In [2]:
free,total=torch.cuda.mem_get_info(); params=70_000_000_000; ideal_int4=params*0.5; reserve=total*0.1; fit=ideal_int4 < total-reserve
w=torch.randn(1024,1024,device=device); x=torch.randn(64,1024,device=device); ref=x@w.t(); q4=symmetric_quantize(w,bits=4,group_size=128)[2]; q8=symmetric_quantize(w,bits=8,group_size=128)[2]
sensitive=torch.arange(0,1024,64,device=device); mixed=q4.clone(); mixed[:,sensitive]=q8[:,sensitive]; err=error_metrics(ref,x@mixed.t())
gates={"single_gpu_ideal_weight_fit":fit,"backend_engine_built":False,"quality_suite_passed":False,"service_slo_passed":False,
       "toy_mixed_bit_rmse_lte_2":err["rmse"]<=2.0,"rollback_artifact_defined":True}
decision="not_ready_for_service" if not all(gates.values()) else "ready_for_canary"
result=base_result(30,"capacity-model"); result.update({"live_gpu_total_gib":round(total/2**30,3),"ideal_int4_weight_gib":round(ideal_int4/2**30,3),
    "toy_mixed_bit_error":err,"deployment_gates":gates,"decision":decision,
    "conclusion":"The gate matrix kept unexecuted 70B engine, quality, and service tests explicit; arithmetic compression alone was insufficient."})


## 3. Inspect the evidence

The notebook can approve further engineering or reject single-GPU feasibility; it cannot claim a 70B engine benchmark without loading one.

### Acceptance and rollback gate

Leave every unexecuted gate visibly false. Require a real 70B load, native operator trace, frozen quality suite, service-load SLO, capacity margin, cost model, canary plan, and tested rollback before deployment.

In [3]:
artifact_path = save_result(result, lesson_dir)
print(json.dumps(result, indent=2, sort_keys=True))
print("Saved: artifacts/rtx5090-result.json")


{
  "conclusion": "The gate matrix kept unexecuted 70B engine, quality, and service tests explicit; arithmetic compression alone was insufficient.",
  "decision": "not_ready_for_service",
  "deployment_gates": {
    "backend_engine_built": false,
    "quality_suite_passed": false,
    "rollback_artifact_defined": true,
    "service_slo_passed": false,
    "single_gpu_ideal_weight_fit": false,
    "toy_mixed_bit_rmse_lte_2": false
  },
  "environment": {
    "compute_capability": "12.0",
    "cuda_runtime": "13.0",
    "gpu": "NVIDIA GeForce RTX 5090",
    "gpu_memory_gib": 31.358,
    "python": "3.12.13",
    "torch": "2.12.0"
  },
  "evidence_label": "capacity-model",
  "executed_at_utc": "2026-08-07T14:46:28+00:00",
  "ideal_int4_weight_gib": 32.596,
  "lesson": 30,
  "live_gpu_total_gib": 31.358,
  "schema_version": 1,
  "toy_mixed_bit_error": {
    "cosine": 0.99336779,
    "mae": 2.96422219,
    "max_abs": 16.94711685,
    "rmse": 3.72017455
  }
}
Saved: artifacts/rtx5090-result.j

## 4. Explain the result

A defensible plan exposes every gate, owner, artifact, and reversal condition before production optimization begins.

Relate the measured fields back to the mechanism above. Treat the checked-in result as one hardware/software observation, not a universal ranking. The complete derivation, evidence boundary, and primary references are in [`README.md`](README.md).